In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 46 samples and 13827 columns.
First few rows of the data:


,progression-free-survival_days,recurrence,overall-survival_days,live/dead,sample_title,tissue,age,Stage,response_to_platinum_therapy,SAMD11,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,3756.0,0.0,3756,0,hgsc_01,high grade serous ovarian carcinoma,45,III,No recurrence,2.622398,...,8.676223,7.069518,8.267111,9.153929,7.122459,7.420737,9.243329,8.370306,7.026580,8.463395
1,NaN,NaN,279,0,hgsc_02,high grade serous ovarian carcinoma,75,III,Not available,1.689550,...,9.223183,7.209203,8.381465,8.653469,6.086647,7.023155,8.876251,7.411800,6.123122,7.926758
2,NaN,NaN,215,0,hgsc_04,high grade serous ovarian carcinoma,43,III,Not available,3.977427,...,8.894088,7.228627,8.621799,9.052544,7.176844,6.796638,9.066246,7.694903,6.928110,8.032013
3,1084.0,0.0,1084,0,hgsc_05,high grade serous ovarian carcinoma,77,III,No recurrence,4.305930,...,8.956461,7.315395,8.491565,9.153493,7.494347,7.630588,9.436103,8.365899,7.008584,8.428227
4,1113.0,0.0,1113,0,hgsc_08,high grade serous ovarian carcinoma,55,III,No recurrence,2.412708,...,9.039857,7.313635,8.498872,8.923264,7.495338,7.567952,9.450154,8.330159,7.004884,8.064721


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for progression-free-survival_days or recurrence: 4
Number of samples with complete data for both variables: 42
Number of censored samples (event_var == 0.0): 8
Number of events (event_var == 1.0): 34
Censored ratio (among complete): 8 / 42 = 0.190 (19.0%)
Number of samples with incomplete data for overall-survival_days or live/dead: 0
Number of samples with complete data for both variables: 46
Number of censored samples (event_var == 0.0): 36
Number of events (event_var == 1.0): 10
Censored ratio (among complete): 36 / 46 = 0.783 (78.3%)
Updated stats for 2 of 2 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE165808/description.json
